# 02 - Create EchoNet segmentation masks

This notebook first creates one binary LV mask for visual validation, then provides the full preprocessing cell that converts all traced frames into image-mask pairs.

In [ ]:
# Kaggle execution order: run after 01_explore_dataset.ipynb has verified paths.
# First run the single-frame sanity check. Only run full preprocessing after the overlay looks correct.
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import cv2
import matplotlib.pyplot as plt

from src.utils import (
    load_echonet_tables,
    preprocess_traced_frames,
    read_video_frame,
    save_mask_sanity_figure,
    tracing_group_to_mask,
    tracing_group_to_polygon,
    video_path_from_name,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "EchoNet-Dynamic"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

file_list, tracings = load_echonet_tables(RAW_DIR)

## Single-mask sanity check

Assumption: each traced frame group contains paired LV border points. The mask is generated by building a closed polygon from `(X1, Y1)` points followed by reversed `(X2, Y2)` points, then filling that polygon at the native frame resolution.

In [ ]:
selected_key = None
selected_rows = None

for key, rows in tracings.groupby(['FileName', 'Frame'], sort=True):
    if video_path_from_name(key[0], RAW_DIR).exists():
        selected_key = key
        selected_rows = rows
        break

assert selected_key is not None, 'No traced frame with an available video was found.'
file_name, frame_idx = selected_key
video_path = video_path_from_name(file_name, RAW_DIR)

frame = read_video_frame(video_path, int(frame_idx))
mask = tracing_group_to_mask(selected_rows, frame.shape[:2])
polygon = tracing_group_to_polygon(selected_rows)

print(f"Selected video: {file_name}")
print(f"Selected traced frame: {frame_idx}")
print(f"Frame shape: {frame.shape}")
print(f"Mask foreground pixels: {(mask > 0).sum():,}")

save_mask_sanity_figure(
    frame,
    mask,
    polygon,
    FIGURES_DIR / 'single_mask_sanity_check.png',
    title=f'{file_name} frame {frame_idx}',
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(frame, cmap='gray')
axes[0].plot(polygon[:, 0], polygon[:, 1], color='yellow', linewidth=1)
axes[0].set_title('Original frame + tracing')
axes[1].imshow(mask, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Binary mask')
axes[2].imshow(frame, cmap='gray')
axes[2].imshow(mask, cmap='Reds', alpha=0.35)
axes[2].set_title('Overlay')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Full preprocessing

Run this section only after the single-mask overlay matches the LV. It writes grayscale frames to `data/processed/images/` and binary masks to `data/processed/masks/`. Existing files are overwritten with deterministic names, so the step is reproducible and easy to rerun.

In [ ]:
# Smoke-test preprocessing configuration for Kaggle: set MAX_SAMPLES = 16.
# Full preprocessing configuration: set MAX_SAMPLES = None.
MAX_SAMPLES = 16

summary = preprocess_traced_frames(
    tracings=tracings,
    raw_dir=RAW_DIR,
    output_dir=PROCESSED_DIR,
    figures_dir=FIGURES_DIR,
    max_samples=MAX_SAMPLES,
    save_examples=8,
)
summary

For a full Kaggle run, change `MAX_SAMPLES` to `None`. The preprocessing summary is also saved to `data/processed/preprocess_summary.json`.